# 🎯 LATS: Language Agent Tree Search

## Learning Objectives
In this notebook, you will learn:
1. **From ToT to LATS** - What Monte Carlo Tree Search adds to "propose and evaluate"
2. **The search tree** - A `Node` that stores a partial solution plus visit counts and accumulated value
3. **Select, expand, evaluate, backpropagate** - The four MCTS steps, with an LLM as proposer and value model
4. **Reflection on failed paths** - Turn dead ends into short lessons that steer later expansions
5. **Measuring it** - Solve rate, LLM calls and iterations used, with every solution checked in code

## Where this fits
`02_tree_of_thoughts.ipynb` searched level by level: every round it expanded the whole frontier and kept the best `BEAM` states. LATS spends the budget **one node at a time**, choosing where to look next from everything learned so far, and it stops as soon as it succeeds. It also borrows the reflection idea from module 11 (`modules/11_reflection/notebooks/02_reflexion_agents.ipynb`): a failed path produces a lesson that later proposals can read.

## Prerequisites
- `02_tree_of_thoughts.ipynb` (the Game of 24 environment, proposer and evaluator are reused here in compact form)
- Dependencies installed with `uv sync` at the repository root
- `OPENAI_API_KEY` in the project-root `.env`

| Property | Value |
|---|---|
| Origin | Zhou et al., *Language Agent Tree Search Unifies Reasoning, Acting, and Planning in Language Models* (2024). [arXiv:2310.04406](https://arxiv.org/abs/2310.04406) |

---
## 📦 Step 1: Environment Setup

Load the API keys from the project-root `.env`. LangSmith tracing is optional: it is used only if your `.env` turns it on.

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: Load API keys and suppress warnings
# ============================================================================
import os
import warnings

from dotenv import find_dotenv, load_dotenv

warnings.filterwarnings("ignore")

# Reads OPENAI_API_KEY from the project-root .env
load_dotenv(find_dotenv(usecwd=True))

# Optional: LangSmith traces every LLM call if the .env sets LANGSMITH_TRACING=true.
# Nothing in this notebook depends on it.
tracing = os.getenv("LANGSMITH_TRACING", os.getenv("LANGCHAIN_TRACING_V2", "false")).lower() == "true"
print(f"✅ Environment ready (LangSmith tracing: {'on' if tracing else 'off'})")

In [ ]:
# ============================================================================
# LLM INITIALIZATION: proposer, value model and reflector
# ============================================================================
import threading

from aied_helpers import get_openai_llm
from langchain_core.callbacks import BaseCallbackHandler


class CallCounter(BaseCallbackHandler):
    """Counts LLM calls so the search budget is visible."""

    def __init__(self):
        self.calls = 0
        self._lock = threading.Lock()

    def on_chat_model_start(self, *args, **kwargs):
        with self._lock:
            self.calls += 1


counter = CallCounter()
proposer_llm = get_openai_llm(model="gpt-4o-mini", temperature=0.7, callbacks=[counter])
judge_llm = get_openai_llm(model="gpt-4o-mini", temperature=0, max_tokens=300, callbacks=[counter])

print(f"✅ LLMs initialized: {proposer_llm.model_name} (proposer T=0.7, value model and reflector T=0)")

---
## 🌲 Part 1: From Tree of Thoughts to LATS

LATS keeps ToT's branching idea — the LLM proposes candidate next steps and a value model scores them — and adds three Monte Carlo Tree Search (MCTS) mechanics:

1. **Selection with UCB1.** Instead of expanding every frontier state every round, LATS walks down from the root picking, at each level, the child with the best `average value + exploration bonus`, and expands only the leaf it reaches.
2. **Backpropagation.** A new child's reward is added to every ancestor's statistics, so a discovery deep in the tree changes which branch looks best near the root on the next iteration.
3. **Reflection.** When a path dead-ends, the LLM writes a one-sentence lesson; the next expansions see the latest lessons in their prompt.

### High-level Workflow
1. **Select** a leaf by UCB1, skipping branches already known to be dead (a node whose children all failed gets one more expansion first).
2. **Expand** it into up to `K` children (LLM proposals, legality checked in code).
3. **Evaluate** each child: exact check in code when two numbers are left, an LLM verdict otherwise.
4. **Backpropagate** the reward to the root; **reflect** on children that dead-ended.
5. Stop at the first solution or when `MAX_ITERATIONS` is used up.

### Strengths & Weaknesses
* **Strengths:** focuses the budget on promising subtrees; stops early on easy problems; rewards and lessons flow between branches.
* **Weaknesses:** more moving parts (value model, exploration constant, bookkeeping); a noisy value model misleads selection; for small problems ToT or even CoT may be enough.

---
## 🎲 Part 2: The Environment and the Search Tree

The Game of 24 environment is the one from `02_tree_of_thoughts.ipynb`: moves are applied with exact fractions, so an illegal or mis-computed LLM proposal is dropped. The `Node` adds the MCTS bookkeeping: `visits`, accumulated `value`, and a `dead` flag for branches that can no longer reach 24.

$$\text{UCB1}(child) = \underbrace{\frac{child.value}{child.visits}}_{\text{exploitation}} + \underbrace{c \cdot \sqrt{\frac{\ln(parent.visits)}{child.visits}}}_{\text{exploration bonus}}$$

An unvisited child gets infinite priority, so every child is tried once before the statistics are trusted.

In [ ]:
# ============================================================================
# ENVIRONMENT (compact version of 02_tree_of_thoughts.ipynb)
# ============================================================================
import math
from dataclasses import dataclass, field
from fractions import Fraction
from typing import List, Literal, Optional

from openai import LengthFinishReasonError
from pydantic import BaseModel, Field


def fmt(x: Fraction) -> str:
    return str(x.numerator) if x.denominator == 1 else f"{x.numerator}/{x.denominator}"


class Move(BaseModel):
    left: str = Field(description="First number, copied exactly from the current list (e.g. '8' or '8/3')")
    op: Literal["+", "-", "*", "/"]
    right: str = Field(description="Second number, copied exactly from the current list")


def apply_move(numbers, move):
    try:
        a, b = Fraction(move.left), Fraction(move.right)
    except (ValueError, ZeroDivisionError):
        return None
    rest = list(numbers)
    for x in (a, b):
        if x not in rest:
            return None
        rest.remove(x)
    if move.op == "/" and b == 0:
        return None
    result = {"+": a + b, "-": a - b, "*": a * b, "/": a / b if b else None}[move.op]
    return rest + [result], f"{fmt(a)} {move.op} {fmt(b)} = {fmt(result)}"


def final_moves_to_24(numbers):
    a, b = numbers
    options = [(a + b, "+", a, b), (a * b, "*", a, b), (a - b, "-", a, b), (b - a, "-", b, a)]
    if b:
        options.append((a / b, "/", a, b))
    if a:
        options.append((b / a, "/", b, a))
    return [f"{fmt(x)} {op} {fmt(y)} = 24" for value, op, x, y in options if value == 24]


# ============================================================================
# SEARCH TREE NODE
# ============================================================================
EXPLORATION_CONSTANT = 1.4
MAX_EXPANSIONS = 2          # a node whose children all failed may be re-expanded once, with the lessons learned


@dataclass
class Node:
    numbers: list
    steps: list = field(default_factory=list)
    parent: Optional["Node"] = None
    children: list = field(default_factory=list)
    visits: int = 0
    value: float = 0.0
    dead: bool = False      # no path from here can reach 24 (as far as the search knows)
    solved: bool = False
    expansions: int = 0     # how many times the proposer was asked for moves from here

    def average_value(self) -> float:
        return self.value / self.visits if self.visits else 0.0

    def ucb1(self) -> float:
        if self.visits == 0:
            return float("inf")
        exploration = EXPLORATION_CONSTANT * math.sqrt(math.log(self.parent.visits) / self.visits)
        return self.average_value() + exploration


root = Node(numbers=[Fraction(n) for n in (4, 7, 8, 8)])
print(f"✅ Root node: {[fmt(x) for x in root.numbers]}")

---
## 🔄 Part 3: Select, Expand, Evaluate, Backpropagate — and Reflect

- **`select`** walks down by UCB1 over *live* children. If every child of a node is dead, the node is expanded once more (up to `MAX_EXPANSIONS`) — this time the prompt carries the lessons from its failed children. After that it is marked dead, so the search never wastes calls there again.
- **`expand`** asks the proposer for `K` new moves (states already in the tree are skipped). The prompt includes the latest reflections.
- **`evaluate`** returns a reward in [0, 1]: two numbers left → exact (1 = solved, 0 = dead); three numbers left → the LLM's `sure / likely / impossible` mapped to 1 / 0.33 / 0.
- **`backpropagate`** adds the reward to the node and every ancestor.
- **`reflect`** turns a dead end into a one-sentence lesson.

In [ ]:
# ============================================================================
# THE FOUR MCTS STEPS + REFLECTION
# ============================================================================
class Proposals(BaseModel):
    moves: List[Move]


class Value(BaseModel):
    attempts: str = Field(description="At most three short attempts, e.g. '4*6=24 but 5 is left over'")
    verdict: Literal["sure", "likely", "impossible"]


proposer = proposer_llm.with_structured_output(Proposals)
value_model = judge_llm.with_structured_output(Value)
REWARD = {"sure": 1.0, "likely": 0.33, "impossible": 0.0}


def select(root: Node) -> Optional[Node]:
    node = root
    while node.children:
        live = [c for c in node.children if not c.dead]
        if not live:
            if node.expansions < MAX_EXPANSIONS:
                return node       # every child failed: ask for new moves here, now with reflections
            node.dead = True      # still nothing: prune this node too
            return None
        node = max(live, key=lambda c: c.ucb1())
    return node


def expand(node: Node, k: int, reflections: list) -> list:
    lessons = "\n".join(f"- {r}" for r in reflections[-3:]) or "(none yet)"
    prompt = (
        f"Game of 24. Current numbers: {[fmt(x) for x in node.numbers]}. Steps so far: {node.steps or '(none)'}.\n"
        f"Lessons from failed attempts:\n{lessons}\n"
        f"Propose up to {k} different next steps. Each step combines exactly two numbers from the list with one of + - * /."
    )
    node.expansions += 1
    seen = {tuple(sorted(c.numbers)) for c in node.children}   # never re-add a state already tried
    new_children = []
    for move in proposer.invoke(prompt).moves:
        result = apply_move(node.numbers, move)
        if result and tuple(sorted(result[0])) not in seen:
            seen.add(tuple(sorted(result[0])))
            new_children.append(Node(numbers=result[0], steps=node.steps + [result[1]], parent=node))
    node.children.extend(new_children)
    if not new_children and node.expansions >= MAX_EXPANSIONS:
        node.dead = True
    return new_children


def evaluate(child: Node) -> float:
    if len(child.numbers) == 2:
        hits = final_moves_to_24(child.numbers)
        if hits:
            child.solved, child.steps = True, child.steps + [hits[0]]
            return 1.0
        child.dead = True
        return 0.0
    try:
        verdict = value_model.invoke(
            f"Game of 24. Can the numbers {[fmt(x) for x in child.numbers]} reach exactly 24, using every "
            "number exactly once with + - * /? Try a few combinations, then answer sure, likely or impossible."
        ).verdict
    except LengthFinishReasonError:  # an evaluation cut off by max_tokens counts as "likely"
        verdict = "likely"
    return REWARD[verdict]


def backpropagate(node: Node, reward: float) -> None:
    while node is not None:
        node.visits += 1
        node.value += reward
        node = node.parent


def reflect(puzzle, child: Node) -> str:
    return judge_llm.invoke(
        f"Game of 24 with {puzzle}. This attempt failed: {child.steps} left {[fmt(x) for x in child.numbers]}, "
        "which cannot make 24. In one short sentence, give a SPECIFIC lesson that names numbers, e.g. "
        "'2*11=22 leaves 6 and 8, which cannot make 2; prefer a first step that leaves a factor pair of 24 like 3 and 8.'"
    ).content.strip()


print("✅ select / expand / evaluate / backpropagate / reflect defined")

---
## 🔁 Part 4: The Search Loop

One iteration = one **select → expand → evaluate → backpropagate** cycle (plus reflections for new dead ends). The loop stops at the first solved child, when the root is dead (nothing left to try), or after `MAX_ITERATIONS`.

In [ ]:
# ============================================================================
# LATS SEARCH LOOP
# ============================================================================
MAX_ITERATIONS = 10
K_CANDIDATES = 4


def print_tree(node: Node, indent: str = "") -> None:
    label = "SOLVED" if node.solved else ("dead" if node.dead else f"avg={node.average_value():.2f}")
    step = node.steps[-1] if node.steps else "(start)"
    print(f"{indent}- {step:<16} {str([fmt(x) for x in node.numbers]):<16} visits={node.visits} {label}")
    for child in node.children:
        print_tree(child, indent + "    ")


def lats(puzzle, max_iterations=MAX_ITERATIONS, k=K_CANDIDATES, verbose=False) -> dict:
    root = Node(numbers=[Fraction(n) for n in puzzle])
    reflections = []
    for iteration in range(1, max_iterations + 1):
        leaf = select(root)
        if leaf is None:
            if root.dead:
                break
            continue
        for child in expand(leaf, k, reflections):
            reward = evaluate(child)
            backpropagate(child, reward)
            if child.solved:
                return {"solution": child.steps, "iterations": iteration, "root": root, "reflections": reflections}
            if child.dead:
                reflections.append(reflect(puzzle, child))
        if verbose:
            print(f"=== iteration {iteration}: expanded {leaf.steps or '(start)'} ===")
    return {"solution": None, "iterations": iteration, "root": root, "reflections": reflections}


counter.calls = 0
run = lats([2, 6, 8, 11])
print_tree(run["root"])
print(f"\nSolution: {run['solution']} | iterations: {run['iterations']} | LLM calls: {counter.calls}")
print("Reflections written:", run["reflections"] or "(none)")

Read the tree: the root was expanded first, every child got one visit, and the next iterations went down the branch with the best average value. Branches marked `dead` are never selected again; their reflections were added to the later expansion prompts — look for nodes that gained extra children on their second expansion. If the run above found a solution quickly, the tree is small — LATS stops at the first success rather than finishing a level.

---
## 📊 Part 5: Measuring It on the ToT Puzzles

The same eight puzzles as `02_tree_of_thoughts.ipynb`. For each run we record whether it was solved, the iterations used (never more than `MAX_ITERATIONS`, asserted below) and the LLM calls. Every solution is a list of steps the environment applied, so it is valid by construction.

In [ ]:
# ============================================================================
# EXPERIMENT: LATS on 8 puzzles (runs concurrently)
# ============================================================================
import time
from concurrent.futures import ThreadPoolExecutor

PUZZLES = [[2, 2, 5, 8], [2, 6, 8, 11], [7, 8, 9, 10], [2, 3, 9, 13],
           [11, 12, 12, 13], [5, 8, 12, 12], [4, 7, 11, 12], [1, 7, 9, 11]]

counter.calls = 0
t0 = time.time()
with ThreadPoolExecutor(max_workers=4) as pool:
    runs = list(pool.map(lats, PUZZLES))
elapsed = time.time() - t0

assert all(r["iterations"] <= MAX_ITERATIONS for r in runs), "the iteration cap must hold"
print(f"{'puzzle':<18}{'solved':>7}{'iterations':>12}   steps")
for p, r in zip(PUZZLES, runs):
    print(f"{str(p):<18}{'✅' if r['solution'] else '❌':>6}{r['iterations']:>10}     {r['solution'] or ''}")
solved = [r for r in runs if r["solution"]]
print(f"\nLATS: {len(solved)}/{len(PUZZLES)} solved, {counter.calls} LLM calls in total, {elapsed:.0f}s")
if solved:
    print(f"Solved runs needed {sum(r['iterations'] for r in solved) / len(solved):.1f} iterations on average")

In our runs LATS solved 2–4 of the 8 puzzles and the ToT beam about 4, with LATS spending more LLM calls, so do not expect LATS to win on this small problem. What the table does show is *how* LATS spends its budget: solved puzzles usually finish in two or three iterations, while unsolved ones use the whole budget or run out of live branches. On problems with a larger branching factor and a better value model — the setting LATS was designed for — that adaptivity is what makes the difference. The levers are the same as for ToT: `MAX_ITERATIONS` and `K_CANDIDATES` trade calls for coverage, and the value model decides where the budget goes.

---
## 📝 Summary

In this notebook, we learned:

### 1. From ToT to LATS
- LATS = ToT's "LLM proposes, LLM evaluates" + MCTS's select → expand → evaluate → backpropagate loop + reflections.

### 2. The search tree
- Each `Node` holds a partial solution plus `visits` and accumulated `value`; UCB1 balances exploitation and exploration.

### 3. The four MCTS steps
- Only one leaf is expanded per iteration; rewards flow back to the root and change the next selection. Dead branches are pruned for good.

### 4. Reflection on failed paths
- A dead end becomes a one-sentence lesson in the next expansion prompts — reflection used *inside* a search.

### 5. Measuring it
- With a capped budget, LATS stops at the first success; on the Game of 24 with gpt-4o-mini it did not beat the simpler ToT beam.

### Next Steps
- **`04_mental_loop_simulator.ipynb`**: search over *actions in an environment* — test a proposed action in a simulator before executing it for real